# AISlopDetector — AI vs Human Generated Dataset

Trains a general-purpose AI image detector on ~100K images spanning multiple
generator families at native resolution. Unlike CIFAKE (32x32, one generator),
this produces a model that generalizes to real-world AI images.

**Before running**: Click "+ Add Data" in the right sidebar and add:
1. `alessandrasala79/ai-vs-human-generated-dataset`

Estimated: ~2 hours on Kaggle T4 GPU

In [ ]:
!pip install -q timm

In [ ]:
import os, random, glob
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import timm
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# === Config ===
DATA_DIR = "/kaggle/input/ai-vs-human-generated-dataset"
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-4

# Explore the dataset structure first
for root, dirs, files in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, "").count(os.sep)
    if depth <= 2:
        jpgs = len([f for f in files if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))])
        if jpgs > 0:
            rel = os.path.relpath(root, DATA_DIR)
            print(f"  {rel}: {jpgs} images")

In [ ]:
# === Build image lists from the actual dataset structure ===

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE + 32),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def collect_images(root, max_per_class=50000):
    """Walk the dataset and collect (path, label) pairs.
    Labels: 0=REAL/human, 1=AI/generated"""
    images = []

    # Try common path patterns
    for split in ["train", "Train", "TRAIN"]:
        for cls_name, label in [("REAL", 0), ("Real", 0), ("real", 0),
                               ("HUMAN", 0), ("Human", 0), ("human", 0),
                               ("AI", 1), ("ai", 1), ("FAKE", 1), ("Fake", 1), ("fake", 1)]:
            path = os.path.join(root, split, cls_name)
            if os.path.isdir(path):
                files = glob.glob(os.path.join(path, "*.*"))
                if len(files) > max_per_class:
                    files = random.sample(files, max_per_class)
                for f in files:
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                        images.append((f, label))

    # Also try flat structure: root/REAL/ and root/AI/
    for cls_name, label in [("REAL", 0), ("Real", 0), ("real", 0),
                           ("HUMAN", 0), ("Human", 0), ("human", 0),
                           ("AI", 1), ("ai", 1), ("FAKE", 1), ("Fake", 1), ("fake", 1)]:
        path = os.path.join(root, cls_name)
        if os.path.isdir(path):
            files = glob.glob(os.path.join(path, "*.*"))
            if len(files) > max_per_class:
                files = random.sample(files, max_per_class)
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    images.append((f, label))

    # Also check nested: root/REAL/*.jpg and root/AI/*.jpg at deeper levels
    if not images:
        # Desperate: scan all subdirs looking for REAL and AI folders
        for dirpath, dirnames, filenames in os.walk(root):
            basename = os.path.basename(dirpath).lower()
            if basename in ["real", "human"]:
                label = 0
            elif basename in ["ai", "fake", "generated"]:
                label = 1
            else:
                continue
            for f in filenames:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    images.append((os.path.join(dirpath, f), label))

    return images

all_images = collect_images(DATA_DIR)
random.shuffle(all_images)

labels = [l for _, l in all_images]
print(f"Total images: {len(all_images)}")
print(f"  REAL: {labels.count(0)}")
print(f"  FAKE: {labels.count(1)}")

In [ ]:
# === PyTorch Dataset ===

class ImageDataset(Dataset):
    def __init__(self, image_list, transform=None):
        self.image_list = image_list
        self.transform = transform

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        path, label = self.image_list[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# 80/10/10 split
n = len(all_images)
train_n = int(n * 0.8)
val_n = int(n * 0.1)
test_n = n - train_n - val_n

train_images = all_images[:train_n]
val_images = all_images[train_n:train_n + val_n]
test_images = all_images[train_n + val_n:]

train_ds = ImageDataset(train_images, train_transform)
val_ds = ImageDataset(val_images, val_transform)
test_ds = ImageDataset(test_images, val_transform)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
# === Model (matches src/models/classifier.py exactly — no key remap needed) ===

class AISlopClassifier(nn.Module):
    def __init__(self, backbone="efficientnet_b3", num_classes=2, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0)
        with torch.no_grad():
            features = self.backbone.eval()(torch.randn(1, 3, 224, 224))
            in_features = features.shape[1]
        self.backbone.train()
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

model = AISlopClassifier().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# === Training ===

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = train_correct = train_total = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} train")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        preds = model(images).argmax(1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{train_correct/train_total:.3f}")

    model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch} val", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    scheduler.step()
    print(f"  train_acc={train_correct/train_total:.4f}  val_acc={val_correct/val_total:.4f}")

    if val_correct / val_total > best_val_acc:
        best_val_acc = val_correct / val_total
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_accuracy": best_val_acc}, "best_model_v2.pth")
        print(f"  -> Saved (acc={best_val_acc:.4f})")

print(f"\nTraining complete. Best val acc: {best_val_acc:.4f}")

In [ ]:
# === Test Evaluation ===
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

ckpt = torch.load("best_model_v2.pth", map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Test"):
        outputs = model(images.to(device))
        probs = torch.softmax(outputs, 1).cpu().numpy()
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, [p[1] for p in all_probs])
cm = confusion_matrix(all_labels, all_preds)

print(f"Test Accuracy:  {acc:.4f}")
print(f"Test F1 Score:  {f1:.4f}")
print(f"Test ROC-AUC:   {auc:.4f}")
print(f"\n  REAL correct: {cm[0,0]}/{cm[0,0]+cm[0,1]}")
print(f"  FAKE correct: {cm[1,1]}/{cm[1,0]+cm[1,1]}")

In [ ]:
from IPython.display import FileLink
FileLink("best_model_v2.pth")